# Skip-Gram word2vec

Pure NumPy implementation of word2vec using skip-gram with negative sampling.

---

## 1. Setup and Imports

In [88]:
import numpy as np
import matplotlib.pyplot as pyplot
from collections import Counter

# Seed for reporducibility
np.random.seed(42)

## 2. Mini Corpus for Testing

Small dataset to verify our implementation before scaling to larger text.

In [89]:
# Mini corpus for testing
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are enemies",
    "the cat and the dog are friends",
    "the black cat chased the small mouse",
    "dogs love to play with their owners",
    "a lazy cat sleeps all day on the sofa"
]

print("Corpus:")
for i, sentence in enumerate(corpus):
    print(f" {i}: {sentence}")

Corpus:
 0: the cat sat on the mat
 1: the dog sat on the log
 2: cats and dogs are enemies
 3: the cat and the dog are friends
 4: the black cat chased the small mouse
 5: dogs love to play with their owners
 6: a lazy cat sleeps all day on the sofa


## 3. Tokenization

Convert sentences into lists of words and build flat word list.

In [90]:
# Tokenize (separates words)
def tokenize(text):
    return text.lower().split()


# Flatten all sentences in one list of words
all_words = []
for sentence in corpus:
    all_words.extend(tokenize(sentence))

print(f"Total words: {len(all_words)}")
print(f"Sample: {all_words[:10]}")

Total words: 47
Sample: ['the', 'cat', 'sat', 'on', 'the', 'mat', 'the', 'dog', 'sat', 'on']


## 4. Vocabulary Building

Create bidirectional mappings: word ↔ index.  
Sorted alphabetically for reproducibility.

In [91]:
# Build vocabulary
word_counts = Counter(all_words)
vocab = sorted(word_counts.keys())

# Create mappings
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()} # Sorts alphabeticaly

vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size}")
print(f"\nWord to Index mapping (first 10):")
for word in list(vocab)[:10]:
    print(f" '{word}' -> {word_to_idx[word]}")

Vocabulary size: 29

Word to Index mapping (first 10):
 'a' -> 0
 'all' -> 1
 'and' -> 2
 'are' -> 3
 'black' -> 4
 'cat' -> 5
 'cats' -> 6
 'chased' -> 7
 'day' -> 8
 'dog' -> 9


## 5. Skip-Gram Training Pairs Generation

Generate (center_word, context_word) pairs using sliding window.  
Each pair represents one training example for the model.

In [92]:
def generate_training_data(corpus, word_to_idx, window_size = 2):
    """
    Generate (center_word_idx, context_word_idx) pairs for skip-gram.

    Args:
        corpus: List of senteces (strings)
        word_to_idx: Dictionary mapping word -> index
        window_size: How many words to look left/right

    Returns:
        pairs: List of tuples (center_idx, context_idx)
    """
    pairs = []

    for sentence in corpus:
        words = tokenize(sentence)
        word_indices = [word_to_idx[word] for word in words]

        for center_pos, center_idx in enumerate(word_indices):
            for offset in range(-window_size, window_size + 1):
                context_pos = center_pos + offset

                if offset == 0:
                    continue

                if context_pos < 0 or context_pos >= len(word_indices):
                    continue
                
                context_idx = word_indices[context_pos]
                pairs.append((center_idx, context_idx))

    return pairs

test_corpus = ["the cat sat on the mat"]
test_pairs = generate_training_data(test_corpus, word_to_idx, window_size=2)

print(f"Generated {len(test_pairs)} training pairs")
print("\nFirst 10 pairs (center_word, context_word):")
for center_idx, context_idx in test_pairs[:10]:
    center_word = idx_to_word[center_idx]
    context_word = idx_to_word[context_idx]
    print(f" ({center_word}, {context_word})")

Generated 18 training pairs

First 10 pairs (center_word, context_word):
 (the, cat)
 (the, sat)
 (cat, the)
 (cat, sat)
 (cat, on)
 (sat, the)
 (sat, cat)
 (sat, on)
 (sat, the)
 (on, cat)


### Generate pairs for full corpus

Now apply to our entire mini corpus to see total training examples.

In [93]:
all_pairs = generate_training_data(corpus, word_to_idx, window_size=2)

print(f"Total training pairs: {len(all_pairs)}")
print(f"Corpus size: {len(all_words)} words")
print(f"Vocabulary size: {vocab_size} unique words")
print(f"\nAverage pairs per word: {len(all_pairs) / len(all_words):.2f}")

Total training pairs: 146
Corpus size: 47 words
Vocabulary size: 29 unique words

Average pairs per word: 3.11
